In [1]:
import os
import pandas as pd
import re
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer, util

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan perangkat: {device}")

In [ ]:
DATASET_PATH = "dataset/"

df_train = pd.read_csv(os.path.join(DATASET_PATH, "train-00000-of-00001.csv"))
df_dev   = pd.read_csv(os.path.join(DATASET_PATH, "dev-00000-of-00001.csv"))
df_test  = pd.read_csv(os.path.join(DATASET_PATH, "test-00000-of-00001.csv"))

keep_labels = ["false", "true"]
label_map = {"false": "FAKE", "true": "REAL"}
label2id = {"FAKE": 0, "REAL": 1}

filtered_dfs = []
for name, df in [("train", df_train), ("dev", df_dev), ("test", df_test)]:
    df_filtered = df[df["label"].isin(keep_labels)].copy().reset_index(drop=True)
    df_filtered["label"] = df_filtered["label"].map(label_map)
    df_filtered["label_id"] = df_filtered["label"].map(label2id)
    
    print(f"{name.capitalize()} -> Awal: {df.shape[0]}, Setelah Filter: {df_filtered.shape[0]}")
    filtered_dfs.append(df_filtered)

df_train, df_dev, df_test = filtered_dfs

In [ ]:
model_sim = SentenceTransformer("indobenchmark/indobert-base-p1")
model_sim.to(device)

In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

llama_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

llama_pipe = pipeline(
    "text-generation",
    model=llama_model,
    tokenizer=tokenizer,
)

In [ ]:
def generate_queries(claim):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Anda adalah AI pemeriksa fakta. Sebelum memberikan pertanyaan pencarian, Anda harus memikirkan konteks klaim tersebut.
Gunakan format:
Pikiran: [Analisis singkat klaim]
1. [Pertanyaan 1]
2. [Pertanyaan 2]
3. [Pertanyaan 3]

Contoh:
Klaim: "Bawang putih bisa menyembuhkan kanker dalam semalam."
Pikiran: Klaim ini berkaitan dengan kesehatan medis yang ekstrem. Saya perlu mencari bukti ilmiah dari lembaga kesehatan resmi.
1. Apakah ada penelitian medis tentang bawang putih menyembuhkan kanker?
2. Apa tanggapan WHO tentang pengobatan kanker dengan bawang putih?
3. Benarkah bawang putih bisa membunuh sel kanker dalam waktu singkat?

<|eot_id|><|start_header_id|>user<|end_header_id|>
Klaim: "{claim}"
<|end_header_id|>"""

    outputs = llama_pipe(prompt, max_new_tokens=250, do_sample=True, temperature=0.6)
    generated_text = outputs[0]['generated_text'][len(prompt):]
    
    queries = [line.strip() for line in generated_text.split('\n') if line.strip() and line[0].isdigit()]
    return queries[:3]

sample_claim = df_train.iloc[0]['claim']
generated_q = generate_queries(sample_claim)

print(f"Klaim: {sample_claim}")
print("Generated Queries:")
for q in generated_q:
    print(f"- {q}")

In [ ]:
def get_best_query(claim, queries):
    if not queries:
        return claim 

    claim_emb = model_sim.encode(claim, convert_to_tensor=True)
    queries_emb = model_sim.encode(queries, convert_to_tensor=True)
    
    cosine_scores = util.cos_sim(claim_emb, queries_emb)[0]
    best_idx = torch.argmax(cosine_scores).item()
    
    return queries[best_idx]

def get_best_evidence(best_query, row):
    evidence_list = [
        str(row['evidence_1']), str(row['evidence_2']), 
        str(row['evidence_3']), str(row['evidence_4']), 
        str(row['evidence_5'])
    ]

    evidence_list = [e for e in evidence_list if len(e) > 5 and e.lower() != 'nan']
    
    if not evidence_list:
        return "Tidak ada bukti yang ditemukan."

    query_emb = model_sim.encode(best_query, convert_to_tensor=True)
    evidences_emb = model_sim.encode(evidence_list, convert_to_tensor=True)

    cosine_scores = util.cos_sim(query_emb, evidences_emb)[0]
    best_idx = torch.argmax(cosine_scores).item()
    
    return evidence_list[best_idx]

In [ ]:
def generate_explanation_fixed(claim, best_evidence):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Anda adalah asisten pemeriksa fakta yang cerdas. Tugas Anda adalah:
1. Menganalisis hubungan antara KLAIM dan BUKTI yang diberikan.
2. Memberikan penjelasan ringkas berdasarkan informasi di dalam BUKTI tersebut.
3. Menentukan apakah klaim tersebut REAL atau FAKE.

CONTOH 1 (FAKE):
Klaim: "Bawang putih bisa menyembuhkan kanker dalam semalam."
Bukti: "Penelitian medis menunjukkan tidak ada bukti ilmiah bawang putih dapat membunuh sel kanker secara instan. Dokter memperingatkan ini adalah hoaks kesehatan."
Pikiran: Bukti secara eksplisit membantah klaim dengan menyebutkan tidak adanya bukti ilmiah dan label hoaks.
Penjelasan: Klaim ini salah menurut pakar medis karena tidak ada dasar ilmiah yang mendukung penyembuhan kanker secara instan menggunakan bawang putih.
Veracity Predicted: FAKE

CONTOH 2 (REAL):
Klaim: "Pemerintah resmi menaikkan harga BBM per 1 September."
Bukti: "Menteri Energi mengumumkan penyesuaian harga BBM bersubsidi yang mulai berlaku efektif pada tanggal 1 September pukul 14.00 WIB."
Pikiran: Bukti mengonfirmasi klaim tersebut melalui pernyataan resmi pemerintah tentang waktu dan detail kenaikan harga.
Penjelasan: Klaim ini benar karena telah dikonfirmasi oleh pernyataan resmi Menteri Energi mengenai jadwal pemberlakuan harga baru.
Veracity Predicted: REAL

FORMAT OUTPUT:
Penjelasan: [Tuliskan inti dari hasil verifikasi secara singkat]
Veracity Predicted: [FAKE/REAL]

<|eot_id|><|start_header_id|>user<|end_header_id|>
Klaim: "{claim}"
Bukti: "{best_evidence}"
Pikiran:<|end_header_id|>"""

    outputs = llama_pipe(
        prompt, 
        max_new_tokens=300, 
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    
    return outputs[0]['generated_text'][len(prompt):].strip()

In [ ]:
df_train = df_train.reset_index(drop=True)
df_dev   = df_dev.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

df_train["source_id"] = df_train.index
df_dev["source_id"]   = df_dev.index
df_test["source_id"]  = df_test.index

df_train["split"] = "train"
df_dev["split"]   = "dev"
df_test["split"]  = "test"

df_train["content"] = df_train["claim"]
df_dev["content"]   = df_dev["claim"]
df_test["content"]  = df_test["claim"]

df_train["time"] = df_train["reviewDate"]
df_dev["time"]   = df_dev["reviewDate"]
df_test["time"]  = df_test["reviewDate"]

In [ ]:
def process_evidence_pipeline(df, split_name):
    df_aug = df.copy()
    
    q1_list, q2_list, q3_list = [], [], []
    best_q_list, best_ev_list = [], []
    rationales, preds, accs = [], [], []

    for index, row in tqdm(df_aug.iterrows(), total=len(df_aug), desc=f"Processing {split_name}"):
        try:
            claim = row['claim']
            gold_label = row['label']

            queries = generate_queries(claim)
            q1 = queries[0] if len(queries) > 0 else ""
            q2 = queries[1] if len(queries) > 1 else ""
            q3 = queries[2] if len(queries) > 2 else ""
            
            best_q = get_best_query(claim, queries)
            best_ev = get_best_evidence(best_q, row)

            raw_output = generate_explanation_fixed(claim, best_ev)
            clean_output = raw_output.replace("assistant", "").strip()
            
            rat_match = re.search(r"Penjelasan:(.*?)Veracity Predicted:", clean_output, re.DOTALL | re.IGNORECASE)
            rationale = rat_match.group(1).strip() if rat_match else clean_output.split("Veracity Predicted:")[0].strip()
  
            pred_match = re.search(r"Veracity Predicted:\s*(REAL|FAKE)", clean_output, re.IGNORECASE)
            prediction = pred_match.group(1).upper() if pred_match else "UNKNOWN"

            accuracy = 1 if prediction == gold_label else 0

            q1_list.append(q1); q2_list.append(q2); q3_list.append(q3)
            best_q_list.append(best_q); best_ev_list.append(best_ev)
            rationales.append(rationale); preds.append(prediction); accs.append(accuracy)

        except Exception as e:
            print(f"Error pada index {index}: {e}")
            q1_list.append(None); q2_list.append(None); q3_list.append(None)
            best_q_list.append(None); best_ev_list.append(None)
            rationales.append(f"Error: {str(e)}")
            preds.append("UNKNOWN")
            accs.append(0)

    df_aug["query_1"] = q1_list
    df_aug["query_2"] = q2_list
    df_aug["query_3"] = q3_list
    df_aug["best_query"] = best_q_list
    df_aug["best_evidence"] = best_ev_list
    df_aug["ev_rationale"] = rationales
    df_aug["ev_pred"] = preds
    df_aug["ev_acc"] = accs
    
    return df_aug

In [ ]:
print("Memulai proses inferensi untuk data TEST...")
df_test_final = process_evidence_pipeline(df_test, "test")